# QKD IR Benchmark — Google Colab CPU Runtime

Runs the Phase 1 benchmark on Colab's CPU runtime (serial — multiprocessing
hangs in Colab notebooks). No GPU needed.

Default config: n=8192, 200 frames/Q ≈ **~40-60 min wall-clock**.

**Workflow:**
1. Cell 1 installs deps + auto-restarts kernel. Skip Cell 1 after restart.
2. Cells 2-3: setup + code-pool generation (~5 min first time).
3. Cells 4-6: parameters + sweeps.
4. Cells 7-9: plots + download.

## Cell 1 — Install deps (auto-restarts kernel)

In [ ]:
import os, sys

SENTINEL = "/content/.deps_installed"

if not os.path.exists(SENTINEL):
    print("=" * 60)
    print("Stage 1/2: pin numpy/scipy/pandas/pyarrow (no transitive deps)")
    print("=" * 60)
    rc = os.system('pip install --force-reinstall --no-deps "numpy<2.1" "scipy<1.14" "pandas<3.0" pyarrow')
    if rc != 0:
        raise RuntimeError(f"Stage 1 install failed (exit {rc})")
    print()
    print("=" * 60)
    print("Stage 2/2: install sequence + its deps (with --no-deps to keep numpy pin)")
    print("=" * 60)
    os.system("pip install --no-deps sequence")
    os.system("pip install --no-deps qutip qutip-qip networkx tqdm matplotlib gmpy2")
    open(SENTINEL, "w").write("done")
    print()
    print("=" * 60)
    print("DONE installing. Restarting kernel automatically...")
    print("After restart, SKIP this cell and run from Cell 2.")
    print("=" * 60)
    os._exit(0)
else:
    print("Deps already installed; skipping. (Delete /content/.deps_installed if you suspect a broken install.)")

## Cell 2 — Clone repo + version check

In [ ]:
import numpy, scipy, pandas, pyarrow
print(f"numpy   : {numpy.__version__}  (expected 2.0.x)")
print(f"scipy   : {scipy.__version__}  (expected 1.13.x or earlier)")
print(f"pandas  : {pandas.__version__}  (expected 2.x)")
print(f"pyarrow : {pyarrow.__version__}")

try:
    import sequence
    from sequence.kernel.timeline import Timeline
    print(f"sequence: OK")
except ImportError as e:
    print(f"\n[!] sequence NOT importable: {e}")
    print("If you see this, re-run Cell 1 (delete /content/.deps_installed first).")
    raise

import os, sys, subprocess

REPO_URL = "https://github.com/alexandrachirita98/qkd-cascade-ldpc.git"
REPO_DIR = "/content/qkd-cascade-ldpc"

if not os.path.exists(REPO_DIR):
    print(f"\nCloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"\nRepo exists; force-syncing to origin/main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

# Set Python AND shell CWD
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Confirm we got the latest commit
last_commit = subprocess.run(
    ["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
    capture_output=True, text=True,
).stdout.strip()
print(f"\nlatest commit: {last_commit}")
print(f"Python CWD   : {os.getcwd()}")
print(f"sys.path[0]  : {sys.path[0]}")

## Cell 3 — Verify / generate LDPC code pool

In [ ]:
import os, sys
REPO_DIR = "/content/qkd-cascade-ldpc"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.codes.storage import list_available

def needs_codes(target_n):
    pool = list_available()
    return sum(1 for n, _ in pool if n == target_n) < 9

# Always check, generate if missing. Use explicit `cd && python` so the
# shell's CWD is correct (notebooks sometimes lose shell CWD).
for n_target in (1024, 8192):
    if needs_codes(n_target):
        print(f"\n[!] missing n={n_target} codes; generating (this takes a few minutes)...")
        rc = os.system(f"cd {REPO_DIR} && python -m src.tools.generate_codes --frame-lengths {n_target}")
        if rc != 0:
            raise RuntimeError(f"code generation failed for n={n_target} (exit {rc})")
    else:
        print(f"n={n_target}: 9/9 codes present, skipping generation")

# Verify final state
from src.codes.storage import list_available
pool = list_available()
print(f"\nFinal pool ({len(pool)} codes):")
for n, r in pool:
    print(f"  n={n:>5}  R={r:.2f}")

## Cell 4 — Sweep parameters

In [ ]:
N             = 8192     # 1024 (fast) or 8192 (canonical)
ALPHA         = 0.15
FRAMES_PER_Q  = 200      # 1000 = paper-grade; 200 = good compromise
SEED          = 42

payload = N - round(ALPHA * N)
QBERS = [round(0.005 * (i + 1), 4) for i in range(20)]

per_frame_s = 0.20 if N >= 8192 else 0.04
n_frames_total = len(QBERS) * FRAMES_PER_Q * 3
est_min = n_frames_total * per_frame_s / 60

print(f"N            = {N}")
print(f"alpha        = {ALPHA}")
print(f"payload/frame= {payload}")
print(f"frames/Q     = {FRAMES_PER_Q}")
print(f"QBER points  = {len(QBERS)} from {QBERS[0]} to {QBERS[-1]}")
print(f"Total frames : {n_frames_total}")
print(f"\nEstimated wall-clock: ~{est_min:.0f} min")

## Cell 5 — QBER sweep (SERIAL — no multiprocessing)

In [ ]:
import pandas as pd
import time
from src.harness import make_cascade, make_mueller, make_borisov, generate_frames

t0 = time.perf_counter()
parts = []
print(f"QBER sweep: {len(QBERS)} points (serial, ~{est_min:.0f} min total)...\n")

for i, q in enumerate(QBERS):
    t_q = time.perf_counter()
    print(f"  [{i+1:>2}/{len(QBERS)}] Q={q:.3f} ... ", end="", flush=True)

    algos = [
        make_cascade(seed=SEED),
        make_mueller(n=N, seed=SEED),
        make_borisov(n=N, alpha=ALPHA, seed=SEED),
    ]
    frames = generate_frames(q, payload, FRAMES_PER_Q, seed=SEED)
    rows = []
    for alg in algos:
        for j, (a, b, true_q) in enumerate(frames):
            res = alg.fn(a, b, q, true_q)
            rows.append({
                "q": q, "alg": alg.name, "frame_idx": j,
                "leakage_bits": res.leakage_bits, "messages": res.messages,
                "iterations": res.iterations, "success": res.success,
                "wall_clock_s": res.wall_clock_s, "true_qber": res.true_qber,
            })
    parts.append(pd.DataFrame(rows))
    print(f"done in {time.perf_counter()-t_q:>5.0f}s  "
          f"(elapsed {(time.perf_counter()-t0)/60:>5.1f}m)")

df_q = pd.concat(parts, ignore_index=True)
df_q.to_parquet("/content/qber_sweep.parquet", index=False)
print(f"\nQBER sweep total: {(time.perf_counter()-t0)/60:.1f}m  |  "
      f"{len(df_q)} records → /content/qber_sweep.parquet")

## Cell 6 — Mismatch sweep

In [ ]:
from src.harness import run_mismatch_sweep, make_cascade, make_mueller, make_borisov
import time

algorithms = [
    make_cascade(seed=SEED),
    make_mueller(n=N, seed=SEED),
    make_borisov(n=N, alpha=ALPHA, seed=SEED),
]

print(f"Mismatch sweep (3 trueQ x 5 deltas x {FRAMES_PER_Q} frames x 3 alg)...\n")
t0 = time.perf_counter()
df_m = run_mismatch_sweep(
    algorithms,
    true_qbers=[0.02, 0.04, 0.06],
    deltas=[-0.02, -0.01, 0.0, 0.01, 0.02],
    n_frames_per_point=FRAMES_PER_Q,
    n_payload=payload,
    seed=SEED,
    progress_callback=lambda m: print(f"  {m}"),
)
out_path = "/content/mismatch_sweep.parquet"
df_m.to_parquet(out_path, index=False)
print(f"\nMismatch sweep total: {(time.perf_counter()-t0)/60:.1f}m  |  "
      f"{len(df_m)} records → {out_path}")

## Cell 7 — Render plots inline

In [ ]:
from src.harness import make_slide_deck
from pathlib import Path
from IPython.display import Image, display, Markdown

out_dir = Path("/content/plots")
paths = make_slide_deck(df_q, df_m, n_payload=payload, out_dir=out_dir)
print(f"rendered {len(paths)} plots → {out_dir}/\n")

for name, p in paths.items():
    display(Markdown(f"### `{name}`"))
    display(Image(str(p)))

## Cell 8 — Summary table

In [ ]:
from src.harness import per_qa_summary
summary = per_qa_summary(df_q, n_payload=payload)
cols = ["alg", "q", "FER", "f", "f_eff", "mean_messages", "mean_wall_ms", "R_sec_per_block"]
display(summary[cols].round(4))

## Cell 9 — Bundle and download

In [ ]:
import shutil, os
stage = "/content/results_staging"
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(stage, exist_ok=True)
shutil.copy("/content/qber_sweep.parquet", stage)
shutil.copy("/content/mismatch_sweep.parquet", stage)
shutil.copytree("/content/plots", os.path.join(stage, "plots"))
zip_path = shutil.make_archive("/content/qkd_sweep_results", "zip", stage)
size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"Created {zip_path} ({size_mb:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("(Not in Colab — grab the file manually from /content/)")